# PhotoGenius AI – Google Colab Test

Run this notebook **on Colab** (Runtime → Change runtime type → **T4 GPU**).  
No local NVIDIA GPU needed; Colab provides the GPU.

**Steps:**
1. Install dependencies (diffusers, torch, transformers, etc.)
2. Load SDXL pipeline on T4 GPU
3. Test image generation with a sample prompt
4. Test LoRA training with 8 sample images
5. Save outputs to Google Drive

Includes error handling and progress bars.

In [ ]:
# 1. Install dependencies
import subprocess
import sys

packages = [
    "torch>=2.0.0",
    "diffusers>=0.30.0",
    "transformers>=4.40.0",
    "accelerate>=0.33.0",
    "safetensors>=0.4.0",
    "peft",
    "tqdm",
    "huggingface_hub",
    "datasets",
    "Pillow",
]

print("Installing:", ", ".join(packages))
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print("✓ All dependencies installed.")
except subprocess.CalledProcessError as e:
    print(f"✗ Install failed: {e}")
    raise

In [ ]:
# 2. Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
    print("✓ Google Drive mounted at", DRIVE_ROOT)
except Exception as e:
    print("✗ Drive mount failed:", e)
    DRIVE_ROOT = "/content"  # fallback to local

In [ ]:
# 3. GPU check (Colab provides T4 – no nvidia-smi needed)
import torch

def check_gpu():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {name} ({mem:.1f} GB)")
        return True
    print("✗ No GPU. Use Runtime → Change runtime type → T4 GPU.")
    return False

if not check_gpu():
    raise RuntimeError("Switch to a GPU runtime and re-run.")

In [ ]:
# 4. Load SDXL pipeline on T4 GPU
from tqdm import tqdm
from diffusers import StableDiffusionXLPipeline
import torch

pipe = None
try:
    with tqdm(desc="Loading SDXL", total=1, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}") as pbar:
        pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        ).to("cuda")
        pbar.update(1)
    print("✓ SDXL pipeline loaded on GPU.")
except Exception as e:
    print(f"✗ Failed to load SDXL: {e}")
    raise

In [ ]:
# 5. Test image generation
import os

PROMPT = "portrait of a person, studio lighting, professional headshot, high quality"
OUT_DIR = f"{DRIVE_ROOT}/photogenius_colab_output"
os.makedirs(OUT_DIR, exist_ok=True)

try:
    with tqdm(desc="Generating image", total=1, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}") as pbar:
        out = pipe(
            prompt=PROMPT,
            num_inference_steps=30,
            guidance_scale=7.5,
        ).images[0]
        pbar.update(1)

    path = os.path.join(OUT_DIR, "test_generation.png")
    out.save(path)
    print(f"✓ Saved: {path}")
    display(out)
except Exception as e:
    print(f"✗ Generation failed: {e}")
    raise

In [ ]:
# 6. Prepare 8 sample images for LoRA training
from pathlib import Path
from datasets import load_dataset
from PIL import Image

TRAIN_DIR = "/content/training_images"
Path(TRAIN_DIR).mkdir(parents=True, exist_ok=True)

# diffusers/dog-example: 5 images of one dog; we use 5 + duplicate 3 → 8
try:
    ds = load_dataset("diffusers/dog-example", split="train")
    for i in tqdm(range(8), desc="Preparing 8 sample images"):
        idx = i % len(ds)
        img = ds[idx]["image"].convert("RGB").resize((512, 512))
        img.save(Path(TRAIN_DIR) / f"img_{i:02d}.jpg")
    print(f"✓ {TRAIN_DIR} has 8 images.")
except Exception as e:
    print(f"✗ Prepare failed: {e}")
    raise

In [ ]:
# 7. LoRA training (DreamBooth LoRA SDXL)
import subprocess
import sys
import os

LORA_OUT = f"{DRIVE_ROOT}/photogenius_colab_output/lora"
os.makedirs(LORA_OUT, exist_ok=True)

# Clone diffusers and install from source (script expects matching version)
if not os.path.exists("/content/diffusers"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/huggingface/diffusers.git", "/content/diffusers"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "/content/diffusers", "-q"], check=True)

script = "/content/diffusers/examples/dreambooth/train_dreambooth_lora_sdxl.py"
if not os.path.exists(script):
    raise FileNotFoundError(f"Training script not found: {script}")

cmd = [
    "accelerate", "launch", "--mixed_precision", "fp16", script,
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
    "--instance_data_dir=" + TRAIN_DIR,
    "--output_dir=" + LORA_OUT,
    "--instance_prompt=a photo of a dog",
    "--resolution=512",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=1",
    "--gradient_checkpointing",
    "--learning_rate=1e-4",
    "--max_train_steps=100",
    "--checkpointing_steps=50",
    "--mixed_precision=fp16",
]

try:
    proc = subprocess.run(cmd, cwd="/content/diffusers")
    if proc.returncode != 0:
        raise RuntimeError(f"LoRA training exited with {proc.returncode}")
    print("✓ LoRA training done. Output:", LORA_OUT)
except Exception as e:
    print(f"✗ LoRA training failed: {e}")
    raise

In [ ]:
# 8. Save outputs to Drive & summary
import shutil
from pathlib import Path

summary = []
try:
    base = Path(OUT_DIR)
    summary.append(f"Output folder: {OUT_DIR}")
    for f in sorted(base.rglob("*")):
        if f.is_file():
            rel = f.relative_to(base)
            summary.append(f"  • {rel} ({f.stat().st_size / 1024:.1f} KB)")
    # Copy training images to output for reference
    train_backup = base / "training_images"
    train_backup.mkdir(exist_ok=True)
    for p in Path(TRAIN_DIR).iterdir():
        if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
            shutil.copy2(p, train_backup / p.name)
    summary.append(f"  • training_images/ (8 sample images)")
except Exception as e:
    summary.append(f"Error: {e}")

print("PhotoGenius Colab – outputs saved to Google Drive\n" + "\n".join(summary))